# Tema 7 — Reconocimiento de Entidades Nombradas (NER)  
## Listado 1 — Notebook resuelto (basado en lo visto en clase)

Este notebook resuelve los ejercicios del listado:

- **Ejercicio 1:** NER con **NLTK** y **spaCy** (modelo *small* vs *transformer*) + NER biomédico con **SciSpaCy**.  
- **Ejercicio 2:** **Fine-tuning** de un modelo NER de spaCy en **español** (`es_core_news_sm`), y ampliación con una **nueva etiqueta** (`JOB`).

> [!important] Nota de entorno (muy común)
> Si al importar `spacy` aparece un error de compatibilidad con `pydantic`, reinstala con:
> ```bash
> pip install -U "pydantic<2" "spacy>=3.7,<3.8"
> ```
> y reinicia el kernel.

---

## 0. Instalación y descargas (ejecutar una vez)

Descomenta y ejecuta las líneas necesarias en tu entorno (Colab / local).

```bash
pip install -U nltk spacy spacy-transformers scikit-learn pandas
python -m spacy download en_core_web_sm
python -m spacy download en_core_web_trf
python -m spacy download es_core_news_sm
```

### SciSpaCy (solo para el apartado biomédico)
SciSpaCy y sus modelos se instalan aparte. Descomenta según tu necesidad:

```bash
pip install -U scispacy spacy
# Modelo biomédico (puede cambiar por versión; ver documentación de scispacy)
pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_jnlpba_md-0.5.4.tar.gz
```

---

In [ ]:
# Descargas NLTK necesarias para NER con ne_chunk
import nltk

nltk.download("punkt")
nltk.download("averaged_perceptron_tagger")
nltk.download("maxent_ne_chunker")
nltk.download("words")

# En algunas versiones recientes también puede hacer falta:
# nltk.download("punkt_tab")


# Ejercicio 1 — NER con NLTK y spaCy

Frase base:
> `"Barack Obama was the 44th president of the United States"`

Se pide:
1. (a) NLTK: `word_tokenize` → `pos_tag` → `ne_chunk` y recorrer el `Tree` (**binary=True vs False**).  
2. (b) spaCy `en_core_web_sm`.  
3. (c) spaCy `en_core_web_trf` (RoBERTa).  
4. (d) Repetir en varias frases y comparar *sm* vs *trf*.  
5. (e) SciSpaCy `en_ner_jnlpba_md` para entidades biomédicas.

---

In [ ]:
# Utilidades para el ejercicio 1
from typing import List, Tuple
from nltk import word_tokenize, pos_tag, ne_chunk
from nltk.tree import Tree

def extract_entities_from_tree(tree: Tree) -> List[Tuple[str, str]]:
    """Extrae entidades desde un nltk.Tree devuelto por ne_chunk.
    Devuelve lista de (texto_entidad, etiqueta).
    """
    entities = []
    for node in tree:
        if isinstance(node, Tree):
            label = node.label()
            text = " ".join([leaf[0] for leaf in node.leaves()])
            entities.append((text, label))
    return entities

def nltk_ner(sentence: str, binary: bool = False) -> List[Tuple[str, str]]:
    tokens = word_tokenize(sentence)
    tagged = pos_tag(tokens)
    tree = ne_chunk(tagged, binary=binary)
    return extract_entities_from_tree(tree)

sentence = "Barack Obama was the 44th president of the United States"

print("NLTK (binary=False):", nltk_ner(sentence, binary=False))
print("NLTK (binary=True) :", nltk_ner(sentence, binary=True))

# Comentario esperado:
# - binary=False devuelve etiquetas específicas (PERSON, GPE, ORGANIZATION, ...)
# - binary=True devuelve una única etiqueta 'NE' para cualquier entidad


In [ ]:
# (b) spaCy con en_core_web_sm
# Si no tienes el modelo, ejecuta: python -m spacy download en_core_web_sm

def spacy_ents(model_name: str, text: str):
    import spacy
    nlp = spacy.load(model_name)
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]

print("spaCy en_core_web_sm:", spacy_ents("en_core_web_sm", sentence))


In [ ]:
# (c) spaCy con en_core_web_trf (Transformer)
# Requiere: pip install spacy-transformers
# Y: python -m spacy download en_core_web_trf

print("spaCy en_core_web_trf:", spacy_ents("en_core_web_trf", sentence))


## (d) Comparación en varias frases

Frases:
- `"Apple is looking at buying U.K. startup for $1 billion"`
- `"The Eiffel Tower is located in Paris, France"`
- `"Cristiano Ronaldo scores winner in Champions League final"`

Compararemos:
- NLTK
- spaCy `en_core_web_sm`
- spaCy `en_core_web_trf`

---

In [ ]:
sentences = [
    "Apple is looking at buying U.K. startup for $1 billion",
    "The Eiffel Tower is located in Paris, France",
    "Cristiano Ronaldo scores winner in Champions League final"
]

import pandas as pd

rows = []
for s in sentences:
    rows.append({
        "sentence": s,
        "NLTK (binary=False)": nltk_ner(s, binary=False),
        "spaCy sm": spacy_ents("en_core_web_sm", s),
        "spaCy trf": spacy_ents("en_core_web_trf", s),
    })

df_compare = pd.DataFrame(rows)
df_compare


### ¿Por qué pueden diferir `sm` vs `trf`?

- `en_core_web_sm` suele ser más **ligero** y puede cometer más **omisiones** o confundir etiquetas.
- `en_core_web_trf` usa Transformers y suele capturar mejor contexto y límites de entidad, pero:
  - es **más lento**
  - requiere más **memoria**
  - puede cambiar la segmentación (p. ej. “U.K.” vs “U.K. startup”).

---

## (e) NER biomédico con SciSpaCy

Frase:
> `"The p53 protein regulates the expression of CDKN1A mRNA in HeLa cells."`

Modelo:
- `en_ner_jnlpba_md`

---

In [ ]:
# SciSpaCy (opcional)
# Si no lo tienes instalado:
# pip install -U scispacy
# pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_jnlpba_md-0.5.4.tar.gz

biomed_text = "The p53 protein regulates the expression of CDKN1A mRNA in HeLa cells."

try:
    import spacy
    nlp_bio = spacy.load("en_ner_jnlpba_md")
    doc_bio = nlp_bio(biomed_text)
    ents_bio = [(ent.text, ent.label_) for ent in doc_bio.ents]
    print("Entidades biomédicas detectadas:")
    for e in ents_bio:
        print(" -", e)
except Exception as e:
    print("No se pudo cargar SciSpaCy/modelo biomédico.")
    print("Error:", repr(e))
    print("Revisa instalación de scispacy y del modelo en_ner_jnlpba_md.")


# Ejercicio 2 — Fine-tuning de spaCy para NER en español

Objetivo:
1. (a) Cargar `es_core_news_sm` y ver entidades detectadas en el artículo.
2. (b) Convertir tuplas de entrenamiento → `spacy.training.Example`.
3. (c) Fine-tuning del componente `ner` (25 iteraciones) y volver a predecir sobre el artículo.
4. (d) Añadir nueva etiqueta `JOB` y reentrenar.

> [!warning] Dataset mínimo
> El entrenamiento es **muy pequeño** (3 ejemplos). Los resultados pueden ser inestables y no generalizar.
> Aun así, sirve para practicar el **pipeline de entrenamiento** en spaCy.

---

In [ ]:
# Texto de la noticia (se usa en varios apartados)
text = """En medio de una cuidada puesta en escena digna del fin de una guerra, y ante más de 50 cámaras de 
medios, algunos venidos de fuera de España para la ocasión, Carmen Cervera, el Ministro de Cultura Miquel Iceta, 
en representación del Gobierno del Estado español, y Borja Thyssen-Bornemisza han procedido esta mañana a 
la dilatada firma del acuerdo que permite la permanencia en España del Mata Mua y una parte importante de la 
colección de la baronesa Thyssen-Bornemisza. El cuadro más famoso de pintor postimpresionista francés Paul 
Gauguin ya cuelga en las salas del museo madrileño. Ha sido una larga travesía en las que se han desarrollado 
intensísimas negociaciones, retiradas de las partes, principios de acuerdo y retrasos desde hace casi una década. 
"Ha sido necesario mucho esfuerzo de las dos partes, pero hoy al fin es una hermosa realidad", ha declarado una 
satisfecha la baronesa. "Hasta hace pocos días no estaba seguro que el cuadro estuviese aquí entre nosotros. Hoy 
tenemos el honor y el privilegio de disfrutar en España de esta y otras pinturas que componen una colección de 
las más importantes del mundo. Esta firma es un final feliz", concluyó el ministro de Cultura. El contrato de 
arrendamiento asegura la permanencia del cuadro en España, junto con 320 obras pertenecientes a la colección 
Carmen Thyseen-Bornemisza, una cuarta parte menos de la garantía actual, durante 15 años. A cambio, la 
baronesa recibirá 6,5 millones de euros anuales en calidad de préstamo. Transcurrido ese tiempo, y pagado el 
importe total, 97,5 millones de euros, el Estado podrá optar a la compra del cuadro, descontando del precio final 
lo pagado. Esto puede suponer reeditar en el futuro los problemas hoy finiquitados."""

In [ ]:
# (a) Entidades con es_core_news_sm
# Si no tienes el modelo: python -m spacy download es_core_news_sm

def spacy_es_ents(nlp, text: str):
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]

import spacy
nlp_es = spacy.load("es_core_news_sm")

ents_before = spacy_es_ents(nlp_es, text)
print("Entidades detectadas (antes de fine-tuning):")
for e in ents_before[:40]:
    print(" -", e)
print(f"\nTotal entidades: {len(ents_before)}")


In [ ]:
# (b) Crear Examples desde el training dado
from spacy.training import Example

training = [
(
"Acuerdo entre el Gobierno del Estado Español y el Gobierno de la República Federal de Alemania para la aplicación del Convenio de 20 de abril de 1966 sobre Seguro de Desempleo.",
[(17, 44, "ORG"), (50, 94, "ORG"), (156, 175, "MISC")]
),
(
"Esta mañana, el primer secretario del PSC, Miquel Iceta, ha tomado posesión del cargo de ministro de Cultura y Deportes.",
[(16, 33, "PER"), (38, 41, "ORG"), (43, 55, "PER"), (89, 119, "PER")]
),
(
"España el país que más gusta a los franceses.",
[(0, 6, "LOC")]
)
]

examples = []
for raw_text, entity_offsets in training:
    ex = Example.from_dict(
        nlp_es.make_doc(raw_text),
        {"entities": entity_offsets}
    )
    examples.append(ex)

print("Examples creados:", len(examples))
print("Ejemplo 1 (texto):", examples[0].text)
print("Ejemplo 1 (entities):", [(e.start_char, e.end_char, e.label_) for e in examples[0].reference.ents])


In [ ]:
# (c) Fine-tuning del componente NER
import random

# Congelar el resto del pipeline excepto NER
disabled_pipes = [pipe for pipe in nlp_es.pipe_names if pipe != "ner"]

optimizer = nlp_es.create_optimizer()

# Entrenamiento sencillo (25 epochs)
with nlp_es.disable_pipes(*disabled_pipes):
    for epoch in range(25):
        random.shuffle(examples)
        losses = {}
        for ex in examples:
            nlp_es.update([ex], sgd=optimizer, losses=losses)
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:02d} | Losses: {losses}")

# Evaluar de nuevo en el artículo
ents_after = spacy_es_ents(nlp_es, text)

print("\nEntidades detectadas (después de fine-tuning):")
for e in ents_after[:40]:
    print(" -", e)
print(f"\nTotal entidades: {len(ents_after)}")


In [ ]:
# Comparación rápida antes vs después (qué entidades nuevas aparecen)
set_before = set(ents_before)
set_after = set(ents_after)

print("Nuevas entidades (después) que no estaban antes (hasta 30):")
for e in list(set_after - set_before)[:30]:
    print(" -", e)

print("\nEntidades que estaban antes y ahora no (hasta 30):")
for e in list(set_before - set_after)[:30]:
    print(" -", e)


## (d) Añadir una nueva etiqueta `JOB` y reentrenar

Se proporciona un nuevo `training` que introduce la categoría **JOB**.
Pasos:
1. Añadir label con `ner.add_label("JOB")`
2. Convertir a `Example`
3. Reentrenar

---

In [ ]:
# Nuevo training con etiqueta JOB
training_job = [
(
"Acuerdo entre el Gobierno del Estado Español y el Gobierno de la República Federal de Alemania para la aplicación del Convenio de 20 de abril de 1966 sobre Seguro de Desempleo.",
[(17, 44, "ORG"), (50, 94, "ORG"), (156, 175, "MISC")]
),
(
"Esta mañana, el primer secretario del PSC, Miquel Iceta, ha tomado posesión del cargo de ministro de Cultura y Deportes.",
[(16, 33, "JOB"), (38, 41, "ORG"), (43, 55, "PER"), (89, 119, "JOB")]
),
(
"España el país que más gusta a los franceses.",
[(0, 6, "LOC")]
)
]

ner = nlp_es.get_pipe("ner")
print("Default labels:", ner.labels)

# Añadir nueva label
ner.add_label("JOB")
print("New labels:", ner.labels)


In [ ]:
# Crear Examples con JOB
examples_job = []
for raw_text, entity_offsets in training_job:
    ex = Example.from_dict(
        nlp_es.make_doc(raw_text),
        {"entities": entity_offsets}
    )
    examples_job.append(ex)

print("Examples JOB:", len(examples_job))
print("Entities ejemplo JOB:", [(e.text, e.label_) for e in examples_job[1].reference.ents])


In [ ]:
# Reentrenar con JOB (mismas condiciones simples)
import random

disabled_pipes = [pipe for pipe in nlp_es.pipe_names if pipe != "ner"]
optimizer = nlp_es.create_optimizer()

with nlp_es.disable_pipes(*disabled_pipes):
    for epoch in range(25):
        random.shuffle(examples_job)
        losses = {}
        for ex in examples_job:
            nlp_es.update([ex], sgd=optimizer, losses=losses)
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:02d} | Losses: {losses}")

# Evaluación de nuevo en el artículo
ents_after_job = spacy_es_ents(nlp_es, text)

print("\nEntidades detectadas (después de entrenar con JOB):")
for e in ents_after_job[:60]:
    print(" -", e)
print(f"\nTotal entidades: {len(ents_after_job)}")


## Comentario final (qué debes observar)

- Tras el primer fine-tuning, deberían cambiar **límites** y/o **tipos** de algunas entidades en el artículo.  
- Tras añadir `JOB`, el modelo podría empezar a marcar expresiones similares a “primer secretario”, “ministro de …” como `JOB`, pero:
  - con tan pocos ejemplos, el comportamiento puede ser **limitado**.

> [!note] Recomendación práctica
> En problemas reales, usa **muchos ejemplos** y separa en `train/val/test`. Aquí se practica el **procedimiento**.

---